# GPU serving optimization experiment
Select NVIDIA T4 and run all. This experiment compares original retention copying, sliced retention copying, and immutable-prefix sharing. Stage profiling is disabled for every timed configuration; all use final-token-only logits. A separate full-logits no-cache reference checks outputs.

Default: original hot trace distribution, seed 99, 32 MiB, 60 requests, 3 timing repeats, concurrency 1. Seven configurations per repeat plus one oracle = 22 runs / 1,320 request executions. Each configuration restarts the service. Progress and checkpoint ZIPs are saved after each configuration. No Drive required. No speedup is assumed.


In [ ]:
import os, sys, subprocess, hashlib, json
from pathlib import Path
from google.colab import files
repo = Path("/content/kv-serving-source")
url = "https://github.com/swathiblrs/Learning-Based-KV-Cache-Management-for-GPU-LLM-Inference.git"
if repo.exists():
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", url, str(repo)], check=True)
os.chdir(repo)
checksums = {str(p.relative_to(repo)): hashlib.sha256(p.read_bytes()).hexdigest()
             for p in (repo / "src").rglob("*.py")}
(repo / "source-manifest.json").write_text(json.dumps(checksums, indent=2))
os.environ["USE_TF"] = "0"
subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26,<21"], check=True)
subprocess.run([sys.executable, "-m", "virtualenv", "--system-site-packages", "--no-download", ".colab-venv"], check=True)
python = str(repo / ".colab-venv/bin/python")
subprocess.run([python, "-m", "pip", "install", "-e", ".[serving]"], check=True)


In [ ]:
subprocess.run([python, "-c", "import torch; assert torch.cuda.is_available(), 'Select GPU runtime'; print(torch.cuda.get_device_name())"], check=True)
subprocess.run([python, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([python, "-m", "kv_cache.cli", "experiment", "--output", "results/experiment"], check=True)


## Matched comparison
Keep PREFIX_TOKENS=0 for the original variable-length workload first. For a separate long-prefix experiment later, choose PREFIX_TOKENS=1024 and BUDGET_MIB=256. Label that as a different workload; a 1024-token prefix takes about 45 MiB in this model/precision and will not fit a 32 MiB budget. Longer prefixes are outside the predictor training-length range.

Change SEED and WORKLOAD in later runs to check generalization. Repeats on one seed measure timing variation only. Do not tune on reported test results.


In [ ]:
CONCURRENCY = 1
WORKLOAD = "hot"
SEED = 99
REQUESTS = 60
REPEATS = 3
BUDGET_MIB = 32
PREFIX_TOKENS = 0
from datetime import datetime, timezone
run_dir = Path("/content") / ("kv-optimization-" + datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S-%f"))
archive = str(run_dir) + ".zip"
command = [python, "-u", "-m", "kv_cache.serve_eval", "--predictor", "results/experiment/predictor.json",
           "--device", "cuda", "--requests", str(REQUESTS), "--repeats", str(REPEATS),
           "--seed", str(SEED), "--workload", WORKLOAD, "--budget-mib", str(BUDGET_MIB),
           "--concurrency", str(CONCURRENCY), "--prefix-tokens", str(PREFIX_TOKENS),
           "--no-profile-stages", "--logits-mode", "last", "--reference-full-logits",
           "--policies", "none", "lru", "learned", "--copy-modes", "baseline", "sliced", "shared",
           "--output", str(run_dir)]
try:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end="", flush=True)
    if process.wait() != 0:
        raise RuntimeError("Evaluation failed; download diagnostics below")
    subprocess.run([python, "-m", "kv_cache.serve_summary", str(run_dir)], check=True)
finally:
    if run_dir.exists():
        import shutil
        shutil.copyfile(repo / "source-manifest.json", run_dir / "source-manifest.json")
        shutil.make_archive(str(run_dir), "zip", run_dir)
    if Path(archive).exists():
        files.download(archive)


In [ ]:
manifest = json.loads((run_dir / "manifest.json").read_text())
print("Suite status:", manifest["status"])
comparison_file = run_dir / "comparison.json"
if comparison_file.exists():
    comparison = json.loads(comparison_file.read_text())
    for row in comparison["groups"]:
        print(row["policy"], row["copy_mode"],
              "median run-p95 ms:", round(row["median_run_p95_ms"], 2),
              "p95 reduction vs no-cache %:", round(row["median_paired_p95_reduction_percent"], 1),
              "faster repeats:", row["faster_than_no_cache_repeats"], "/", row["repeats"])
    print(comparison["note"])
else:
    print("Download diagnostics; do not bypass failed checks.")


## Download again if needed
This cell only downloads the most recent ZIP. It does not rerun the experiment. If the runtime was reset, an undownloaded archive may no longer exist.


In [ ]:
from pathlib import Path
from google.colab import files
archives = sorted(Path("/content").glob("kv-optimization-*.zip"))
if archives:
    files.download(str(archives[-1]))
else:
    print("No serving archive found in this runtime.")
